# Session 3 — Prompts and structured outputs

**Goal:** make model output consumable by software: schema, strict parsing, and refusal that survives validation.

In [ ]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
CORPUS_DIR = REPO_ROOT / "data" / "corpus"

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
    print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    LIVE = preflight(REPO_ROOT)  # the configured lane's client; FakeLLM whenever the lane is down

In [ ]:
from bootcamp_agent.checks import check, review

## 1. Unconstrained vs typed

The same 'model', two contracts. Prose is for people. JSON with a fixed shape is for software.

In [ ]:
import json

from bootcamp_agent.llm import FakeLLM

prose_llm = FakeLLM(
    default="Chunking is, broadly speaking, quite useful, and many practitioners agree."
)
typed_llm = FakeLLM(
    default=json.dumps(
        {
            "answer": "Chunking splits documents into retrievable passages.",
            "citations": ["rag-basics"],
            "confidence": 0.85,
            "needs_human_review": False,
        }
    )
)

question = "How does chunking work?"
print("PROSE:", prose_llm.complete(system="", user=question))
print("TYPED:", typed_llm.complete(system="", user=question))

## 2. Parsing is an application responsibility

`parse_research_answer` rejects missing fields, unknown fields, non-JSON, and out-of-range confidence. The model's output is untrusted input.

In [ ]:
from bootcamp_agent.schema import AnswerParseError, parse_research_answer

good = parse_research_answer(typed_llm.complete(system="", user=question))
print(good)

## 3. Exercise: sneak something past the parser

**Context.** A schema only protects you if violations fail loudly. Try to get three different bad payloads through.

**Instructions.**

1. Attempt 1 is done: prose, not JSON.
2. Attempt 2: valid JSON that misses one required field.
3. Attempt 3: all four fields present, but `confidence` outside 0.0 to 1.0.
4. Run the cell: every attempt must print `rejected`. Then run the check.

In [ ]:
attempts = [
    "The answer is chunking.",  # attempt 1, done: prose is not JSON
    "",  # TODO(you): valid JSON with a missing field
    "",  # TODO(you): all four fields, confidence out of range
]

for attempt in attempts:
    try:
        parse_research_answer(attempt)
        print(f"ACCEPTED: {attempt[:60]!r}")
    except AnswerParseError as error:
        print(f"rejected: {error}")

**Expected output** (yours may differ in wording, not in shape):

```
rejected: Not valid JSON: Expecting value: line 1 column 1 (char 0)
rejected: Wrong fields: missing=['confidence', 'needs_human_review'] unknown=[]
rejected: 'confidence' out of range [0, 1]: 7
✅ ch03-e1 passed
```

In [ ]:
check("ch03-e1", attempts)

## 4. Exercise: three golden questions

**Context.** A golden set is the smallest evaluation there is: a question and the behaviour a correct assistant shows. Author three against `data/corpus/`.

**Instructions.**

1. The **answerable** case is done: the corpus clearly supports it.
2. Write the **ambiguous** case: two documents could plausibly answer it.
3. Write the **unsupported** case: the corpus says nothing about it.
4. For each, `expected_behavior` says what a correct assistant does. Run the check: it retrieves each question and confirms the kind.

In [ ]:
golden = [
    {
        "question": "What stopping conditions should an agent loop have?",  # done
        "kind": "answerable",
        "expected_behavior": "answers, citing agent-loops",
    },
    {"question": "", "kind": "ambiguous", "expected_behavior": ""},  # TODO(you)
    {"question": "", "kind": "unsupported", "expected_behavior": ""},  # TODO(you)
]
for case in golden:
    print(f"{case['kind']:12} {case['question'] or '(empty)'}")

**Expected output** (yours may differ in wording, not in shape):

```
answerable   What stopping conditions should an agent loop have?
ambiguous    How do I keep an assistant safe?
unsupported  What is the best pizza in Sao Paulo?
✅ ch03-e2 passed
```

In [ ]:
check("ch03-e2", golden)

## 5. The real agent, on the fake lane

The agent refuses *before* calling the model when retrieval finds nothing. Watch the trace prove it.

In [ ]:
from bootcamp_agent.agent import answer_question
from bootcamp_agent.documents import load_corpus

documents = load_corpus(CORPUS_DIR)
for case in golden:
    result = answer_question(case["question"], documents, FakeLLM())
    answer = result.answer
    print(f"\n[{case['kind']}] {case['question']}")
    print(f"  citations={list(answer.citations)} review={answer.needs_human_review}")
    for event in result.trace:
        print(f"  trace[{event.kind}] {event.detail[:80]}")

## 6. Exercise: the real agent, on your lane

**Context.** On the fake lane the model returns a canned refusal. On the ollama lane a real 7B model has to produce the JSON contract, and the corrective retry in `agent.py` may fire. Either way, what reaches you went through the parser.

**Instructions.**

1. The cell runs the answerable question through `FakeLLM()`. Change it to `LIVE`.
2. Read the trace: count the `llm_call` events. Two means the retry fired.
3. Run the check. It confirms the answer came through the parser, on any lane.

In [ ]:
result = answer_question(golden[0]["question"], documents, FakeLLM())  # TODO(you): use LIVE
live_answer = result.answer

print(live_answer.answer)
print(f"citations={list(live_answer.citations)} confidence={live_answer.confidence}")
for event in result.trace:
    print(f"  trace[{event.kind}] {event.detail[:80]}")

**Expected output** (yours may differ in wording, not in shape):

```
An agent loop should stop on a budget of tool calls, on a final answer, or on a refusal ...
citations=['agent-loops'] confidence=0.8
  trace[retrieve] top_k=3 -> [('agent-loops', 1), ('agent-loops', 0), ('agent-loops', 2)]
  trace[llm_call] attempt 1: 212 chars
  trace[decision] answered with citations ['agent-loops']
✅ ch03-e3 passed
```

In [ ]:
check("ch03-e3", live_answer)

## Exit ticket

Homework: add two adversarial questions (e.g. one instructing the assistant to ignore its rules) and compare unstructured vs structured behavior.

## Review

The scorecard for this notebook. Every ❌ line names the exercise and the hint.

In [ ]:
review("ch03")